# KFP hyperparameter tuning: Katib vs custom ParallelFor

This notebook compares two approaches to hyperparameter optimization in Kubeflow Pipelines:
the **Katib** managed service (native hyperparameter tuning with search algorithms)
and a **custom ParallelFor** loop that iterates over a grid of parameters manually.

Both approaches solve the same problem — finding good hyperparameters for a model — but
they differ in setup, scalability, and the features they offer. The goal is to see where
each fits and what you trade off when you pick one over the other.

## Prerequisites

- A Kubeflow cluster with KFP v2 and Katib installed
- `kubeflow-pipeline-sdk` (`kfp`) >= 2.0
- `kubeflow-katib` SDK (`katib`) — usually comes with Kubeflow or install via `pip install kubeflow-katib`

If Katib is not installed on the cluster, the Katib section will fail with a `NotFound` error.
The ParallelFor section only requires KFP.

In [ ]:
import kfp
from kfp import dsl
from kfp import client as kfp_client
import json
import sys

KFP_ENDPOINT = "http://ml-pipeline.kubeflow.svc.cluster.local:8888"
EXPERIMENT_NAME = "hp-tuning-comparison"

print(f"kfp version: {kfp.__version__}")

---
## Part 1 — Custom ParallelFor hyperparameter tuning

This approach uses `dsl.ParallelFor` to iterate over a pre-defined grid of hyperparameter
combinations. Each combination runs a training component in parallel, and the results are
collected after all iterations complete.

**Strengths:** no extra service needed, fully deterministic grid, full control over parallelisation.
**Weaknesses:** you must define the grid up front; no early stopping; no intelligent search.

In [ ]:
from kfp.dsl import component, InputPath, OutputPath, Metrics

@component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn", "pandas", "numpy"],
)
def train_evaluate_component(
    learning_rate: float,
    max_depth: int,
    n_estimators: int,
    metrics_output: OutputPath("Metrics"),
) -> float:
    """Train a simple model with given hyperparams and return validation accuracy."""
    import numpy as np
    from sklearn.datasets import load_iris
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score

    data = load_iris()
    X, y = data.data, data.target

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth if max_depth > 0 else None,
        random_state=42,
    )

    scores = cross_val_score(model, X, y, cv=3, scoring="accuracy")
    mean_acc = float(np.mean(scores))

    import json
    with open(metrics_output, "w") as f:
        json.dump({"accuracy": mean_acc}, f)

    return mean_acc

In [ ]:
@dsl.pipeline(
    name="parallelfor-hp-tuning",
    description="Grid search hyperparameter tuning with ParallelFor",
)
def parallelfor_hp_pipeline():
    """Pipeline that iterates over a hyperparameter grid using ParallelFor."""
    param_grid = [
        {"learning_rate": 0.01, "max_depth": 3, "n_estimators": 50},
        {"learning_rate": 0.01, "max_depth": 5, "n_estimators": 100},
        {"learning_rate": 0.1,  "max_depth": 3, "n_estimators": 50},
        {"learning_rate": 0.1,  "max_depth": 5, "n_estimators": 100},
    ]

    tasks = []
    with dsl.ParallelFor(param_grid) as params:
        task = train_evaluate_component(
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            n_estimators=params["n_estimators"],
        )
        tasks.append(task)

    # One way to surface results is to log them via a collector task.
    # For simplicity here, each task's output is accessible in the UI.


# Compile and verify the pipeline compiles without errors
try:
    compiler = kfp.v2.compiler.Compiler()
    compiler.compile(
        pipeline_func=parallelfor_hp_pipeline,
        package_path="parallelfor_hp_pipeline.yaml",
    )
    print("ParallelFor pipeline compiled successfully.")
except Exception as e:
    print(f"Compilation failed: {e}", file=sys.stderr)
    raise

### Upload and run the ParallelFor pipeline

The cell below uploads the compiled pipeline and creates a run. If the cluster is not
reachable, the client will raise a connection error.

In [ ]:
def run_pipeline_on_cluster(pipeline_path: str, experiment_name: str, run_name: str):
    """Upload a compiled pipeline and start a run."""
    try:
        client = kfp_client.Client(host=KFP_ENDPOINT)
    except Exception as e:
        print(f"Cannot connect to KFP at {KFP_ENDPOINT}: {e}")
        print("Skipping cluster submission. The pipeline YAML was saved locally.")
        return

    try:
        experiment = client.create_experiment(name=experiment_name)
        run = client.run_pipeline(
            experiment_id=experiment.id,
            job_name=run_name,
            pipeline_package_path=pipeline_path,
        )
        print(f"Run submitted: {run_name} (id: {run.id})")
    except Exception as e:
        print(f"Failed to submit run: {e}")


run_pipeline_on_cluster(
    "parallelfor_hp_pipeline.yaml",
    EXPERIMENT_NAME,
    "parallelfor-hp-tuning-run",
)

---
## Part 2 — Katib hyperparameter tuning

Katib is Kubeflow's native hyperparameter tuning service. Instead of a static grid,
you define a search space and Katib chooses hyperparameter combinations using a search
algorithm (random search, grid search, Bayesian optimisation, etc.). Katib can also
early-stop poorly performing trials automatically.

The Katib approach defines an `Experiment` custom resource. This example uses the
Katib SDK (`katib`) to create one programmatically.

In [ ]:
try:
    from kubeflow.katib import KatibClient
    from kubeflow.katib.api import (
        Experiment,
        SearchSpace,
        SearchSpaceParameter,
        ObjectiveSpec,
        AlgorithmSpec,
        TrialSpec,
    )
    KATIB_AVAILABLE = True
except ImportError:
    KATIB_AVAILABLE = False
    print("katib SDK not installed. Install with: pip install kubeflow-katib")
    print("The Katib cells below will define the experiment config but require the SDK to run.")

In [ ]:
# Define the Katib Experiment configuration

KATIB_EXPERIMENT_NAME = "katib-hp-tuning"
KATIB_NAMESPACE = "kubeflow"

# Search space — two parameters to tune
search_space = SearchSpace(
    parameters=[
        SearchSpaceParameter(
            name="n_estimators",
            parameter_type="int",
            feasible_space={"min": "50", "max": "200"},
        ),
        SearchSpaceParameter(
            name="max_depth",
            parameter_type="int",
            feasible_space={"min": "3", "max": "10"},
        ),
    ]
)

# Objective — maximise validation accuracy
objective = ObjectiveSpec(
    type="maximize",
    objective_metric_name="accuracy",
    additional_metric_names=[],
)

# Algorithm — random search (simplest, no dependencies)
algorithm = AlgorithmSpec(
    algorithm_name="random",
)

# Trial spec — the training container
trial_spec = TrialSpec(
    max_trial_count=6,
    parallel_trial_count=2,
    success_condition="status.conditions.#.type==Complete#",
    failure_condition="status.conditions.#.type==Failed#",
    metrics_collector_spec={
        "source": {"file_system_path": {"path": "/output/metrics.json"}},
        "collector": {"kind": "FileCollector"},
    },
)

experiment_config = Experiment(
    name=KATIB_EXPERIMENT_NAME,
    namespace=KATIB_NAMESPACE,
    search_space=search_space,
    objective=objective,
    algorithm=algorithm,
    trial_spec=trial_spec,
)

print("Katib Experiment config defined.")
print(f"  Algorithm: {algorithm.algorithm_name}")
print(f"  Max trials: {trial_spec.max_trial_count}")
print(f"  Search space: n_estimators=[50, 200], max_depth=[3, 10]")

In [ ]:
# Submit the Katib Experiment

if KATIB_AVAILABLE:
    try:
        katib_client = KatibClient(namespace=KATIB_NAMESPACE)
        katib_client.create_experiment(experiment_config)
        print(f"Katib experiment '{KATIB_EXPERIMENT_NAME}' created.")
    except Exception as e:
        print(f"Failed to create Katib experiment: {e}")
        print("Check that Katib is installed on the cluster.")
else:
    print("Katib SDK not available — skipping submission.")
    print("To run this, install kubeflow-katib and ensure Katib is deployed on the cluster.")

In [ ]:
# Monitor Katib experiment status

if KATIB_AVAILABLE:
    import time

    def poll_experiment(name, namespace, timeout_minutes=10):
        client = KatibClient(namespace=namespace)
        start = time.time()
        while time.time() - start < timeout_minutes * 60:
            exp = client.get_experiment(name, namespace)
            status = exp.status.conditions[-1].type if exp.status.conditions else "Pending"
            print(f"  Status: {status}")
            if status == "Succeeded":
                print("Experiment completed.")
                # Get best trial
                trials = client.get_trials(name, namespace)
                best = None
                for t in trials:
                    acc = t.status.metrics[-1].value if t.status.metrics else None
                    if acc and (best is None or acc > best["accuracy"]):
                        best = {"trial": t.metadata.name, "accuracy": acc}
                if best:
                    print(f"  Best trial: {best['trial']} (accuracy: {best['accuracy']})")
                return
            elif status == "Failed":
                print(f"Experiment failed: {exp.status.conditions[-1].message}")
                return
            time.sleep(30)
        print("Timeout reached. Check experiment status manually.")

    poll_experiment(KATIB_EXPERIMENT_NAME, KATIB_NAMESPACE)
else:
    print("Katib SDK not available — cannot poll experiment status.")

---
## Comparison: Katib vs ParallelFor

| Aspect | ParallelFor | Katib |
|--------|-------------|-------|
| **Setup** | Just KFP components and `dsl.ParallelFor` | Requires Katib service installed on the cluster |
| **Search strategy** | You define the exact grid | Katib offers random, grid, Bayesian, TPE, etc. |
| **Early stopping** | None — all combos run to completion | Katib can stop poor trials early (median stop, etc.) |
| **Config scope** | Stays in your pipeline code | Separate Experiment CR — decoupled from pipeline logic |
| **Scalability** | Good for small grids (< 50 combos) | Designed for larger search spaces with smarter sampling |
| **Result tracking** | Manual collection in pipeline | Katib UI + experiment status, best trial reported automatically |
| **Maturity** | KFP built-in, battle-tested | Kubeflow sub-project, stable but may lag on upgrades |

**When to use ParallelFor:** small, known grids (e.g., 4–12 combinations) where you want
all results visible inside a single KFP run. Also useful when you need deterministic
reproducibility of the exact parameter set tested.

**When to use Katib:** larger or unknown search spaces where you want algorithmic search
(Bayesian, random sampling), early stopping, and automatic best-trial reporting. Katib
also handles retries and failure recovery per trial.

One consideration: Katib uses a separate Experiment CR that runs outside the pipeline
DAG. This means the pipeline submits an Experiment and must poll for completion, adding
complexity to the pipeline code compared to the self-contained ParallelFor approach.

---
## Verify

Check that the ParallelFor pipeline compiles correctly and that the Katib experiment
configuration is valid JSON-serialisable (required for the Kubernetes API).

In [ ]:
# Verify 1: ParallelFor pipeline compilation
import os

if os.path.exists("parallelfor_hp_pipeline.yaml"):
    size = os.path.getsize("parallelfor_hp_pipeline.yaml")
    print(f"[PASS] ParallelFor pipeline YAML exists ({size} bytes)")
else:
    print("[FAIL] ParallelFor pipeline YAML not found — compilation may have failed")

# Verify 2: Katib experiment config is serialisable
try:
    cfg = experiment_config.to_dict()
    roundtrip = json.dumps(cfg, indent=2)
    print(f"[PASS] Katib experiment config serialises to JSON ("
          f"{len(json.loads(roundtrip))} top-level keys)")
except Exception as e:
    print(f"[FAIL] Katib config serialisation error: {e}")

## Summary

- **ParallelFor** is the simpler, self-contained approach — no extra dependencies,
  all logic inside one pipeline definition. Best for small, known grids.
- **Katib** adds a managed tuning layer with smarter search, early stopping, and
  built-in best-trial tracking. Requires Katib to be installed on the cluster.
- The two are not mutually exclusive: you can use ParallelFor for a quick baseline
  grid and Katib for a follow-up refinement pass.

The Katib SDK API (`kubeflow.katib`) is still evolving — the constructor signatures
changed between v0.14 and v0.16. If the experiment creation above fails with an
attribute error, check the installed version with `pip show kubeflow-katib` and
consult the [official Katib SDK docs](https://www.kubeflow.org/docs/components/katib/).